# 01a. 채널 정보 수집 — `channels.list` API

**목적:** youtube_channels_cleaned.csv 의 head(5) 샘플 채널에 대해  
YouTube Data API v3 `channels.list` 로 받을 수 있는 **모든 필드**를 확인한다.

| 표시 | 의미 |
|------|------|
| ⭐ MUST | 이탈 예측 피처 또는 영상 수집에 필수 |
| ○ OPTIONAL | 있으면 좋지만 없어도 됨 |

**API Quota 비용:** `channels.list` 1회 = **1 unit** (50채널 배치 가능)

## 0. 설정

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY')
assert API_KEY, '.env 파일에 YOUTUBE_API_KEY 를 설정하세요'

youtube = build('youtube', 'v3', developerKey=API_KEY)

ROOT     = Path('../../')          # 프로젝트 루트
CSV_PATH = ROOT / 'data' / 'raw' / 'youtube_channels_cleaned.csv'
OUT_DIR  = ROOT / 'data' / 'raw' / 'channels'
JSON_DIR = OUT_DIR / 'json'        # 원시 API 응답 저장
CSV_DIR  = OUT_DIR / 'csv'         # MUST 필드 가공본
JSON_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 처리할 youtube_channels_cleaned.csv 의 행 인덱스 범위 (둘 다 inclusive)
# 예: START_IDX=0, END_IDX=49 → 0번째 ~ 49번째 행 (총 50개 채널) 처리
# 다음 실행 때는 직전 실행이 출력한 안내 메시지의 인덱스부터 이어서 처리
START_IDX = 0
END_IDX   = 49
# START_IDX = 250
# END_IDX   = 399
# ─────────────────────────────────────────────────────────────

print('API 연결 완료')

# python notebooks/01_data_collection/run_channel_batch.py --start 2000 --end 3999

API 연결 완료


## 1. 처리 대상 채널 불러오기 (`START_IDX` ~ `END_IDX`)

In [146]:
df_all = pd.read_csv(CSV_PATH)
df = df_all.iloc[START_IDX:END_IDX + 1].copy()

print(f'전체 채널 수: {len(df_all):,}')
print(f'이번 실행 범위: {START_IDX} ~ {END_IDX} (총 {len(df)}개 채널)')
df[['channel_name', 'youtube_channel_id']].head(10)

전체 채널 수: 8,192
이번 실행 범위: 800 ~ 849 (총 50개 채널)


,channel_name,youtube_channel_id
800,부산희야TV(Korean Ghost Tracker),UCPvt1dbhi23-FTpGYZrmpLw
801,기타치는다람띠,UCgXzwGNQLKQ4LA4XmWYWuvw
802,낚튜브 _NotTube,UC7ddGHlf5sUc3w3yeMRqTLA
803,스타토이,UCJ2jmRmeLWFtVQQPsut9v8w
804,방랑이고싶다 BangLang,UCS_GfKlHjjFwgIqGyZOwbtA
805,じゃぴ,UCktQ-dRsgT6IYjKXqw7fTCQ
806,Online Topik,UCG6jmnh5Qw2SZYIPsN2Azfw
807,천재아시안,UCa98sUKOO-0ZoqmnOlY1i2g
808,kpop playlist,UCKFY1KpeayRWa1QUzk-d0zQ
809,BROKENPASTEL,UC66duPrHKb_TJge0IL8x_9g


In [147]:
channel_ids = df['youtube_channel_id'].tolist()
print('수집할 channel_id:')
for cid in channel_ids:
    print(' ', cid)

수집할 channel_id:
  UCPvt1dbhi23-FTpGYZrmpLw
  UCgXzwGNQLKQ4LA4XmWYWuvw
  UC7ddGHlf5sUc3w3yeMRqTLA
  UCJ2jmRmeLWFtVQQPsut9v8w
  UCS_GfKlHjjFwgIqGyZOwbtA
  UCktQ-dRsgT6IYjKXqw7fTCQ
  UCG6jmnh5Qw2SZYIPsN2Azfw
  UCa98sUKOO-0ZoqmnOlY1i2g
  UCKFY1KpeayRWa1QUzk-d0zQ
  UC66duPrHKb_TJge0IL8x_9g
  UC5p-SkOSlkZRiF6CRdIFJvw
  UCopSrNAWJnzm4ETOS4mXHnw
  UCx7ThnRlyrSSiNCLIZI0aqQ
  UCW53Pedlt62rx5zuTRZKUow
  UC-MAo6JvPw49wWJzTePGF5Q
  UCfxGrUvr10MjaCVKE_HVGgw
  UCKcwMdtuEELC4hn_fUlfVrA
  UC9mpnu1x96y7utLpnYyU6kw
  UChC6A-YfVfQ9jbqG5zb4gjw
  UCUEAfMdSYiXDfRVxdfqAQrg
  UCGqrzELBIpH6kfgJsNcZgeQ
  UC0CN1kbW3RuSz60bE9ELHcw
  UCPQ887rCYjOSSxN6nXqAojg
  UC-Id9P16Dnutun3wual1fcA
  UCc8RR9yJEKVvP31OWBqhIjg
  UCHHmDl_k97jLSOYbKEflr5Q
  UCGGhcGI2JzKmtpSUkqSm7HQ
  UCfVqqxdFMvN4f0-isg0_bbw
  UCS7_uGiAocfn0JGrJliGVrA
  UC7SSJS5huELNkaI6XhoNpNw
  UCFHt4Drs8lmyd-aUsBqM5uQ
  UC76R9DL1NjD7aLr_C3eueLA
  UCA7WtDf9bYuRW1BowR00e6g
  UCsnakEva6XIr8rf0P8R_vBQ
  UCCtKt-yPCrhLd6nlgJQc4RQ
  UC3QWksal35X_nSXE8jC59_A
  UCYhXoMjqK

## 2. `channels.list` — 받을 수 있는 모든 part 호출

| Part | 주요 내용 |
|------|-----------|
| `snippet` | 채널명·설명·개설일·국가·썸네일 |
| `contentDetails` | ⭐ uploads 플레이리스트 ID (영상 수집 필수) |
| `statistics` | ⭐ 구독자 수·총 조회수·총 영상 수 |
| `topicDetails` | 콘텐츠 카테고리 토픽 |
| `brandingSettings` | 채널 키워드·배너 |
| `status` | 공개 상태·성인 인증 여부 |

In [148]:
PARTS = 'snippet,contentDetails,statistics,topicDetails,brandingSettings,status'
BATCH_SIZE = 50   # channels.list API 한 번에 최대 50개 ID

QUOTA_REASONS = {'quotaExceeded', 'rateLimitExceeded', 'dailyLimitExceeded', 'userRateLimitExceeded'}

channel_ids_all = df['youtube_channel_id'].tolist()
all_items = []                       # 누적된 channels.list 응답 items
last_completed_idx = START_IDX - 1   # 마지막으로 성공한 df_all 기준 인덱스
api_error = None                     # quota 등 에러 메시지

for batch_start in range(0, len(channel_ids_all), BATCH_SIZE):
    batch_ids   = channel_ids_all[batch_start:batch_start + BATCH_SIZE]
    range_lo    = START_IDX + batch_start
    range_hi    = START_IDX + batch_start + len(batch_ids) - 1
    print(f'  batch {batch_start // BATCH_SIZE + 1}: df_all[{range_lo}..{range_hi}] ({len(batch_ids)}개)')
    try:
        resp = youtube.channels().list(
            part=PARTS,
            id=','.join(batch_ids),
            maxResults=BATCH_SIZE,
        ).execute()
        all_items.extend(resp.get('items', []))
        last_completed_idx = range_hi
    except HttpError as e:
        reason = e.error_details[0]['reason'] if e.error_details else f'HTTP {e.resp.status}'
        api_error = reason
        print(f'  ✗ API 에러: {reason} — 이 배치 이후 중단')
        if reason not in QUOTA_REASONS:
            print('  (quota 에러는 아니지만 안전을 위해 중단합니다)')
        break

response = {'items': all_items}   # 이후 셀에서 사용하는 변수와 호환
n_units  = (last_completed_idx - START_IDX + 1 + BATCH_SIZE - 1) // BATCH_SIZE if last_completed_idx >= START_IDX else 0

print(f'\n수집 결과 — 응답 {len(all_items)}개 / 사용 quota: 약 {n_units} unit')
print(f'성공한 마지막 인덱스: {last_completed_idx}')
if api_error:
    print(f'에러 발생: {api_error}')

  batch 1: df_all[800..849] (50개)

수집 결과 — 응답 49개 / 사용 quota: 약 1 unit
성공한 마지막 인덱스: 849


## 3. 전체 응답 구조 확인

In [149]:
# 첫 번째 채널의 전체 응답을 그대로 출력
sample = response['items'][0]
print(json.dumps(sample, indent=2, ensure_ascii=False))

{
  "kind": "youtube#channel",
  "etag": "sfC3_tEWOW8ucgPl17yiCuamm5s",
  "id": "UCfxGrUvr10MjaCVKE_HVGgw",
  "snippet": {
    "title": "해븐마니아",
    "description": "'해븐마니아'는 공연에 특화된 서비스와 마케팅을 전담하는 브랜드이자,\n당사의 공연을 사랑해 주시는 마니아분들들을 지칭합니다.\n\n해븐마니아 분들을 위한 다양한 공연 콘텐츠를 제공해 드리겠습니다 :)\n많은 관심과 사랑 부탁드립니다!\n",
    "customUrl": "@heavenmania",
    "publishedAt": "2015-01-22T07:45:37Z",
    "thumbnails": {
      "default": {
        "url": "https://yt3.ggpht.com/55KJ50QbibFs9uau5nBklKkYrSgbXq1YQr3mdszpDSQDh2DfJxnYMDnvkLZv5PKxNEp87rtjKg=s88-c-k-c0x00ffffff-no-rj",
        "width": 88,
        "height": 88
      },
      "medium": {
        "url": "https://yt3.ggpht.com/55KJ50QbibFs9uau5nBklKkYrSgbXq1YQr3mdszpDSQDh2DfJxnYMDnvkLZv5PKxNEp87rtjKg=s240-c-k-c0x00ffffff-no-rj",
        "width": 240,
        "height": 240
      },
      "high": {
        "url": "https://yt3.ggpht.com/55KJ50QbibFs9uau5nBklKkYrSgbXq1YQr3mdszpDSQDh2DfJxnYMDnvkLZv5PKxNEp87rtjKg=s800-c-k-c0x00ffffff-no-rj",
        "width": 800,

## 4. 필드별 상세 확인

### 4-1. snippet (기본 정보)

In [150]:
rows = []
for item in response['items']:
    s = item.get('snippet', {})
    rows.append({
        # ⭐ MUST — 채널 식별
        'channel_id':        item['id'],
        'title':             s.get('title'),            # ⭐ MUST  채널명
        'published_at':      s.get('publishedAt'),       # ⭐ MUST  채널 개설일 → 채널 나이 피처
        # ○ OPTIONAL
        'description':       s.get('description'),       # ○ 채널 설명
        'custom_url':        s.get('customUrl'),          # ○ @핸들
        'country':           s.get('country'),            # ○ 국가 코드
        'default_language':  s.get('defaultLanguage'),   # ○ 기본 언어
        'thumbnail_default': s.get('thumbnails', {}).get('default', {}).get('url'),
        'thumbnail_high':    s.get('thumbnails', {}).get('high', {}).get('url'),
    })

pd.DataFrame(rows)

,channel_id,title,published_at,description,custom_url,country,default_language,thumbnail_default,thumbnail_high
0,UCfxGrUvr10MjaCVKE_HVGgw,해븐마니아,2015-01-22T07:45:37Z,"'해븐마니아'는 공연에 특화된 서비스와 마케팅을 전담하는 브랜드이자,\n당사의 공연...",@heavenmania,None,None,https://yt3.ggpht.com/55KJ50QbibFs9uau5nBklKkY...,https://yt3.ggpht.com/55KJ50QbibFs9uau5nBklKkY...
1,UCFHt4Drs8lmyd-aUsBqM5uQ,나반장,2013-08-26T00:49:17Z,.\n\n,@nabanjang,KR,None,https://yt3.ggpht.com/721WXcieL5KhhgZ8_fYImMOH...,https://yt3.ggpht.com/721WXcieL5KhhgZ8_fYImMOH...
2,UCG6jmnh5Qw2SZYIPsN2Azfw,Online Topik,2020-08-10T09:34:29.926778Z,Let's Learn Korean Language \nAll about Korea\...,@onlinetopik267,None,mn,https://yt3.ggpht.com/GhqtCI13KlpEWK2YWRYMLOoK...,https://yt3.ggpht.com/GhqtCI13KlpEWK2YWRYMLOoK...
3,UC5p-SkOSlkZRiF6CRdIFJvw,TOY AMIGO,2015-12-15T07:21:32Z,TOY (장난감) + AMiGO (친구를 뜻하는 스페인어)\n\n\n스톱모션 촬영으...,@toyamigo,KR,ko,https://yt3.ggpht.com/ytc/AIdro_nqZnX5EX5dmiDl...,https://yt3.ggpht.com/ytc/AIdro_nqZnX5EX5dmiDl...
4,UC76R9DL1NjD7aLr_C3eueLA,이상한 나라의 신기한 개발자,2018-12-25T11:04:20Z,재미 있개 풀어가는 개발 이야기 : 질문해서 잔소리 듣고 싶은곳 메일 : wears...,@이상한나라의신기한개,KR,None,https://yt3.ggpht.com/ytc/AIdro_nhdZUiV_8nkjuf...,https://yt3.ggpht.com/ytc/AIdro_nhdZUiV_8nkjuf...
5,UCktQ-dRsgT6IYjKXqw7fTCQ,じゃぴ,2012-10-28T04:32:01Z,こんにちは。ショート動画を中心に投稿しています。♥️🌷\n,@j_lee-y3m,JP,None,https://yt3.ggpht.com/E-3jrZbZPTDrW19QG_aN0FYe...,https://yt3.ggpht.com/E-3jrZbZPTDrW19QG_aN0FYe...
6,UCz0mNYCcfvetgrS7h0H-WcQ,柯柯yeliz,2012-07-07T15:27:39Z,HELLO!! 我是柯柯YELIZ\n喜歡分享我的生活，希望你們喜歡我的生活唷~\n👉🏻合作...,@yeliz5246,None,zh-TW,https://yt3.ggpht.com/ytc/AIdro_kk6zdssPkCdv9P...,https://yt3.ggpht.com/ytc/AIdro_kk6zdssPkCdv9P...
7,UC-Id9P16Dnutun3wual1fcA,출동 도시의 영웅들 (한국말),2014-10-30T13:39:02Z,도시의 영웅들에 오신 것을 환영합니다. \n\n도시의 친구들과 신나는 모험을 떠나...,@출동도시의영웅들한국,None,None,https://yt3.ggpht.com/ytc/AIdro_lEyDtlNsjf10Be...,https://yt3.ggpht.com/ytc/AIdro_lEyDtlNsjf10Be...
8,UC20IAd2RgxMkSNAbiRVMdGQ,한국어 노래 가사 - KTrot 365,2024-10-11T09:27:04.152027Z,"안녕하세요 트롯을 사랑하는 여러분! 오늘은 여러분의 마음을 사로잡을 특별한 영상, ...",@ktrot365,KR,None,https://yt3.ggpht.com/0GX4v0oNUokCeZYD0wuOmz_V...,https://yt3.ggpht.com/0GX4v0oNUokCeZYD0wuOmz_V...
9,UCGqrzELBIpH6kfgJsNcZgeQ,마즐래 WGworking,2012-11-07T16:32:36Z,재활용 나무젓가락으로 공예를 하고 있습니다.\nRecycled wood chopst...,@wgworking,KR,ko,https://yt3.ggpht.com/zLynQDtAgzLrDbm9asq6d067...,https://yt3.ggpht.com/zLynQDtAgzLrDbm9asq6d067...


### 4-2. contentDetails — ⭐ uploads 플레이리스트 ID (영상 수집 핵심)

In [151]:
rows = []
for item in response['items']:
    cd = item.get('contentDetails', {}).get('relatedPlaylists', {})
    rows.append({
        'channel_id':       item['id'],
        'uploads_playlist': cd.get('uploads'),   # ⭐ MUST  영상 목록 수집에 필수
        'likes_playlist':   cd.get('likes'),      # ○ 좋아요 재생목록
    })

df_content = pd.DataFrame(rows)
df_content

,channel_id,uploads_playlist,likes_playlist
0,UCfxGrUvr10MjaCVKE_HVGgw,UUfxGrUvr10MjaCVKE_HVGgw,
1,UCFHt4Drs8lmyd-aUsBqM5uQ,UUFHt4Drs8lmyd-aUsBqM5uQ,
2,UCG6jmnh5Qw2SZYIPsN2Azfw,UUG6jmnh5Qw2SZYIPsN2Azfw,
3,UC5p-SkOSlkZRiF6CRdIFJvw,UU5p-SkOSlkZRiF6CRdIFJvw,
4,UC76R9DL1NjD7aLr_C3eueLA,UU76R9DL1NjD7aLr_C3eueLA,
5,UCktQ-dRsgT6IYjKXqw7fTCQ,UUktQ-dRsgT6IYjKXqw7fTCQ,
6,UCz0mNYCcfvetgrS7h0H-WcQ,UUz0mNYCcfvetgrS7h0H-WcQ,
7,UC-Id9P16Dnutun3wual1fcA,UU-Id9P16Dnutun3wual1fcA,
8,UC20IAd2RgxMkSNAbiRVMdGQ,UU20IAd2RgxMkSNAbiRVMdGQ,
9,UCGqrzELBIpH6kfgJsNcZgeQ,UUGqrzELBIpH6kfgJsNcZgeQ,


### 4-3. statistics — ⭐ 구독자·조회수·영상 수

In [152]:
rows = []
for item in response['items']:
    st = item.get('statistics', {})
    rows.append({
        'channel_id':              item['id'],
        'subscriber_count':        st.get('subscriberCount'),         # ⭐ MUST  구독자 수
        'view_count':              st.get('viewCount'),                # ⭐ MUST  총 채널 조회수
        'video_count':             st.get('videoCount'),               # ⭐ MUST  총 영상 수
        'hidden_subscriber_count': st.get('hiddenSubscriberCount'),   # ○ 구독자 수 비공개 여부
    })

pd.DataFrame(rows)

,channel_id,subscriber_count,view_count,video_count,hidden_subscriber_count
0,UCfxGrUvr10MjaCVKE_HVGgw,6200,3596981,514,False
1,UCFHt4Drs8lmyd-aUsBqM5uQ,6470,227365,22,False
2,UCG6jmnh5Qw2SZYIPsN2Azfw,6190,269277,207,False
3,UC5p-SkOSlkZRiF6CRdIFJvw,6170,8474110,146,False
4,UC76R9DL1NjD7aLr_C3eueLA,6500,1367378,1032,False
5,UCktQ-dRsgT6IYjKXqw7fTCQ,6060,27698,9,False
6,UCz0mNYCcfvetgrS7h0H-WcQ,6740,275423,10,False
7,UC-Id9P16Dnutun3wual1fcA,6300,6380185,98,False
8,UC20IAd2RgxMkSNAbiRVMdGQ,7640,912187,66,False
9,UCGqrzELBIpH6kfgJsNcZgeQ,6280,1120420,129,False


### 4-4. topicDetails (카테고리)

In [153]:
rows = []
for item in response['items']:
    td = item.get('topicDetails', {})
    rows.append({
        'channel_id':       item['id'],
        'topic_ids':        td.get('topicIds'),           # ○ Freebase 토픽 ID 목록
        'topic_categories': td.get('topicCategories'),   # ○ Wikipedia 카테고리 URL
    })

pd.DataFrame(rows)

,channel_id,topic_ids,topic_categories
0,UCfxGrUvr10MjaCVKE_HVGgw,"[/m/019_rr, /m/04rlf, /m/02jjt, /m/05qjc]",[https://en.wikipedia.org/wiki/Lifestyle_(soci...
1,UCFHt4Drs8lmyd-aUsBqM5uQ,"[/m/025zzc, /m/0bzvm2, /m/03hf_rm]","[https://en.wikipedia.org/wiki/Action_game, ht..."
2,UCG6jmnh5Qw2SZYIPsN2Azfw,"[/m/019_rr, /m/01k8wb]",[https://en.wikipedia.org/wiki/Lifestyle_(soci...
3,UC5p-SkOSlkZRiF6CRdIFJvw,"[/m/03glg, /m/019_rr]","[https://en.wikipedia.org/wiki/Hobby, https://..."
4,UC76R9DL1NjD7aLr_C3eueLA,[/m/098wr],[https://en.wikipedia.org/wiki/Society]
5,UCktQ-dRsgT6IYjKXqw7fTCQ,"[/m/0403l3g, /m/0bzvm2, /m/025zzc]",[https://en.wikipedia.org/wiki/Role-playing_vi...
6,UCz0mNYCcfvetgrS7h0H-WcQ,"[/g/120yrv6h, /m/019_rr]","[https://en.wikipedia.org/wiki/Tourism, https:..."
7,UC-Id9P16Dnutun3wual1fcA,"[/m/019_rr, /m/02vxn, /m/02jjt]",[https://en.wikipedia.org/wiki/Lifestyle_(soci...
8,UC20IAd2RgxMkSNAbiRVMdGQ,"[/m/04rlf, /m/028sqc]","[https://en.wikipedia.org/wiki/Music, https://..."
9,UCGqrzELBIpH6kfgJsNcZgeQ,"[/m/019_rr, /m/03glg]",[https://en.wikipedia.org/wiki/Lifestyle_(soci...


### 4-5. brandingSettings

In [154]:
rows = []
for item in response['items']:
    ch = item.get('brandingSettings', {}).get('channel', {})
    img = item.get('brandingSettings', {}).get('image', {})
    rows.append({
        'channel_id':             item['id'],
        'keywords':               ch.get('keywords'),               # ○ 채널 키워드 (태그)
        'unsubscribed_trailer':   ch.get('unsubscribedTrailer'),    # ○ 구독 전 트레일러 영상 ID
        'banner_url':             img.get('bannerExternalUrl'),      # ○ 채널 배너 이미지 URL
    })

pd.DataFrame(rows)

,channel_id,keywords,unsubscribed_trailer,banner_url
0,UCfxGrUvr10MjaCVKE_HVGgw,None,_ZU49MsUnfg,None
1,UCFHt4Drs8lmyd-aUsBqM5uQ,None,None,https://yt3.googleusercontent.com/Zcbo_X0LGT8i...
2,UCG6jmnh5Qw2SZYIPsN2Azfw,None,None,https://yt3.googleusercontent.com/0ampQiH2e15a...
3,UC5p-SkOSlkZRiF6CRdIFJvw,토이아미고(toyamigo),d4UIq1LkFGw,https://yt3.googleusercontent.com/NSZe0SrYQBqC...
4,UC76R9DL1NjD7aLr_C3eueLA,이상한 나라의 신기한 개발자가 사는 세상 이야기 취업,ym6qR-yZZjY,https://yt3.googleusercontent.com/5RhV9q4wYdAV...
5,UCktQ-dRsgT6IYjKXqw7fTCQ,None,DQ7rpcDwKXA,None
6,UCz0mNYCcfvetgrS7h0H-WcQ,美妝 生活 美食 旅遊,None,https://yt3.googleusercontent.com/dPfg21R9DIxj...
7,UC-Id9P16Dnutun3wual1fcA,출동 도시의 영웅들,Gg-DVcGFkyU,https://yt3.googleusercontent.com/X0VgiBgphPGp...
8,UC20IAd2RgxMkSNAbiRVMdGQ,"트로트 트로트메들리 인기트로트 trot ktrot ""내일은 미스트롯"" 젊은트롯 ""요...",None,https://yt3.googleusercontent.com/HF4kFycnzw6I...
9,UCGqrzELBIpH6kfgJsNcZgeQ,DIY 총만들기 미니어처 나무공예 조각 공예 장인 만들기 무기 미니어쳐 수제 수공예...,x15ahhDNG-M,https://yt3.googleusercontent.com/zvj-QkaYKZAD...


### 4-6. status

In [155]:
rows = []
for item in response['items']:
    s = item.get('status', {})
    rows.append({
        'channel_id':          item['id'],
        'privacy_status':      s.get('privacyStatus'),       # ○ public / private / unlisted
        'is_linked':           s.get('isLinked'),             # ○ Google 계정 연결 여부
        'long_uploads_status': s.get('longUploadsStatus'),   # ○ 15분 초과 업로드 가능 여부
        'made_for_kids':       s.get('madeForKids'),         # ○ 어린이용 채널 여부
    })

pd.DataFrame(rows)

,channel_id,privacy_status,is_linked,long_uploads_status,made_for_kids
0,UCfxGrUvr10MjaCVKE_HVGgw,public,True,longUploadsUnspecified,None
1,UCFHt4Drs8lmyd-aUsBqM5uQ,public,True,longUploadsUnspecified,None
2,UCG6jmnh5Qw2SZYIPsN2Azfw,public,True,longUploadsUnspecified,None
3,UC5p-SkOSlkZRiF6CRdIFJvw,public,True,longUploadsUnspecified,None
4,UC76R9DL1NjD7aLr_C3eueLA,public,True,longUploadsUnspecified,False
5,UCktQ-dRsgT6IYjKXqw7fTCQ,public,True,longUploadsUnspecified,False
6,UCz0mNYCcfvetgrS7h0H-WcQ,public,True,longUploadsUnspecified,None
7,UC-Id9P16Dnutun3wual1fcA,public,True,longUploadsUnspecified,True
8,UC20IAd2RgxMkSNAbiRVMdGQ,public,True,longUploadsUnspecified,False
9,UCGqrzELBIpH6kfgJsNcZgeQ,public,True,longUploadsUnspecified,False


## 5. 이탈 예측에 사용할 필드 정리

| 필드 | Part | 용도 |
|------|------|------|
| `channel_id` | — | 키 |
| `published_at` | snippet | ⭐ 채널 나이 계산 |
| `country` | snippet | ○ 국가 피처 |
| `uploads_playlist` | contentDetails | ⭐ 영상 수집 (01b 노트북) |
| `subscriber_count` | statistics | ⭐ 규모 피처 |
| `view_count` | statistics | ⭐ 총 조회수 |
| `video_count` | statistics | ⭐ 총 영상 수 |

## 6. 결과 저장

In [156]:
# 실제로 성공한 범위
actual_end_idx = last_completed_idx
total = len(df_all)

if actual_end_idx < START_IDX or not response['items']:
    print('⚠ 한 건도 수집하지 못함 — CSV 저장을 생략합니다.')
    print(f'▶ 다음 실행 시 START_IDX = {START_IDX} 로 다시 시도하세요.')
else:
    # 원시 API 응답 전체 저장
    out_raw = JSON_DIR / f'channels_raw_{START_IDX}_{actual_end_idx}.json'
    with open(out_raw, 'w', encoding='utf-8') as f:
        json.dump(response['items'], f, indent=2, ensure_ascii=False)
    print(f'원시 저장: {out_raw}')

    # MUST 필드 추출
    must_rows = []
    for item in response['items']:
        s  = item.get('snippet', {})
        cd = item.get('contentDetails', {}).get('relatedPlaylists', {})
        st = item.get('statistics', {})
        must_rows.append({
            'channel_id':       item['id'],
            'title':            s.get('title'),
            'published_at':     s.get('publishedAt'),
            'country':          s.get('country'),
            'uploads_playlist': cd.get('uploads'),
            'subscriber_count': st.get('subscriberCount'),
            'view_count':       st.get('viewCount'),
            'video_count':      st.get('videoCount'),
        })

    df_must = pd.DataFrame(must_rows)
    out_csv = CSV_DIR / f'channels_must_{START_IDX}_{actual_end_idx}.csv'
    df_must.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f'MUST 필드 저장: {out_csv} ({len(df_must)}개 행)')

    # 다음 실행 안내
    if actual_end_idx < END_IDX:
        print(f'\n⚠ API 에러로 일부만 처리됨 — 요청 {START_IDX}~{END_IDX} / 성공 {START_IDX}~{actual_end_idx}')
        print(f'▶ 다음 실행 시 START_IDX = {actual_end_idx + 1} 부터 다시 시작하세요.')
    else:
        print(f'\n✓ 요청 범위 {START_IDX}~{END_IDX} 전체 처리 완료')
        if actual_end_idx + 1 < total:
            remaining = total - actual_end_idx - 1
            print(f'▶ 다음 실행 시 START_IDX = {actual_end_idx + 1} (전체 {total}개 중 {remaining}개 남음)')
        else:
            print(f'▶ youtube_channels_cleaned.csv 전체 처리 완료 (총 {total}개)')

    df_must.head()

원시 저장: ../../data/raw/channels/json/channels_raw_800_849.json
MUST 필드 저장: ../../data/raw/channels/csv/channels_must_800_849.csv (49개 행)

✓ 요청 범위 800~849 전체 처리 완료
▶ 다음 실행 시 START_IDX = 850 (전체 8192개 중 7342개 남음)
